In [1]:
#BILIOTECAS 
import math  #Calculos matematicos
from copy import deepcopy  #Hacer copias profundas de listas
from typing import List , Tuple #Anotaciones de tipo
import random
import os
import time
import sys

In [2]:
#Clase tablero
class Tablerowumpus:
    
    #Constructor
    def __init__(self, matrix=None):
        if matrix is None:
            self.inicializar_tablero()
        else:
            self.setMatrix(matrix)
            
    #Metodo equals
    def __eq__(self, other) -> bool:
        #Verificamos si el otro objeto es una instancia del TableroWumpus
        if not isinstance(other, Tablerowumpus):
            return False
        #Si efectivamente es una instancia: se comparan las matrices de ambos
        return self.matrix == other.matrix
    
    #Metodo para establecer una copia profunda de la matriz del tablero
    def setMatrix(self, matrix):
        self.matrix = deepcopy(matrix)
    
    #Metodo para obtener una copia profunda de la matriz del tablero
    def getMatrix(self) -> List[List]:
        return deepcopy(self.matrix)
    
    def placeTile(self, row: int, col: int, tile: int):
        #Colocamos la ficha en una posicion fila y columna de la matriz
        #Usamos fila-1 y col-1 porque en Python las listas empiezan
        #en 0 pero es mas comodo para nosotros pensar en filas y col empezando en 1
        self.matrix[row-1][col-1] = tile
    
    #Metodo que calcula la distancia
    def distancia(self, pos1, pos2):
        if pos1 is None or pos2 is None:
            return float('inf')
        return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])
    
    #Función de utilidad
    def utility(self, pos_agente: list, pos_oro: list) -> float:
        
        epsilon = 0.000001  # Un pequeño valor para evitar divisiones por cero

        # Calcular la distancia al oro
        distancia_oro = self.distancia(pos_agente, pos_oro)

        # Calcular la distancia al Wumpus
        pos_wumpus = self.encontrar_elemento(3)
        distancia_wumpus = self.distancia(pos_agente, pos_wumpus) if pos_wumpus else float('inf')

        # Calcular las distancias a todos los hoyos
        hoyos = self.encontrar_elementos([4])
        distancias_hoyos = [self.distancia(pos_agente, hoyo) for hoyo in hoyos]

        # Calcular la utilidad 
        utilidad_oro = 1 / (distancia_oro + epsilon)
        utilidad_wumpus = 1 / (distancia_wumpus + epsilon)
        utilidad_hoyos = sum(1 / (dist + epsilon) for dist in distancias_hoyos)

        # La utilidad final es la diferencia entre la atracción al oro y la repulsión de los peligros
        utilidad = utilidad_oro - (utilidad_wumpus + utilidad_hoyos)

        return utilidad
    
    #Metodo que recorre el tablero y reconoce donde deben ser puestos los avisos en el tablero
    def identificaVecinos(self):
        filas = len(self.matrix)
        columnas = len(self.matrix[0])
        
        for i in range(filas):
            for j in range(columnas):
                if self.es_elemento(self.matrix[i][j], 3):  # Wumpus
                    self.marcarVecinos(i, j, 5)  # 5 representa el hedor
                elif self.es_elemento(self.matrix[i][j], 4):  # Hoyo
                    self.marcarVecinos(i, j, 6)  # 6 representa la brisa

    #Funcion que pone en las casillas adyacentes al Wumpus y a los hoyos los distintos avisos de peligro
    def marcarVecinos(self, fila, columna, marca):
        direcciones = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        for dx, dy in direcciones:
            nueva_fila, nueva_columna = fila + dx, columna + dy
            if 0 <= nueva_fila < len(self.matrix) and 0 <= nueva_columna < len(self.matrix[0]):
                casilla_actual = self.matrix[nueva_fila][nueva_columna]
                if marca == 0:  # Si estamos quitando avisos
                    if casilla_actual in [6, 7]:  
                        self.matrix[nueva_fila][nueva_columna] = 0 
                else:  
                    if casilla_actual == 0: 
                        self.matrix[nueva_fila][nueva_columna] = marca
                    elif casilla_actual == 2: 
                        self.matrix[nueva_fila][nueva_columna] = 8 if marca == 5 else 9
                    elif casilla_actual == 5 and marca == 6: 
                        self.matrix[nueva_fila][nueva_columna] = 7
                    elif casilla_actual == 6 and marca == 5: 
                        self.matrix[nueva_fila][nueva_columna] = 7
                    elif casilla_actual == 8 and marca == 6: 
                        self.matrix[nueva_fila][nueva_columna] = 10
                    elif casilla_actual == 9 and marca == 5: 
                        self.matrix[nueva_fila][nueva_columna] = 10

    # Método adicional para imprimir el tablero 
    def printTablero(self):
        for fila in self.matrix:
            print(' '.join(map(str, fila)))
        print()
    
    #Metodo que inicializa el tablero segun las reglas
    def inicializar_tablero(self):
        while True:
            # Inicializar tablero vacío
            self.matrix = [[0 for _ in range(6)] for _ in range(6)]
            
            # Colocar el agente en la esquina inferior izquierda
            self.matrix[5][0] = 1
            
            # Colocar el oro (posición fija)
            self.colocar_oro()
            
            # Guarda la posición del oro
            self.pos_oro = (1,2)
            
            # Colocar el Wumpus aleatoriamente
            if not self.colocar_aleatoriamente(3):
                continue
            
            # Colocar dos hoyos aleatoriamente
            hoyos_colocados = 0
            for _ in range(2):
                if self.colocar_aleatoriamente(4):
                    hoyos_colocados += 1
            
            if hoyos_colocados < 2:
                continue
            
            # Generar hedores y brisas
            self.identificaVecinos()
            break 
    
    #Metodo para colocar aleatoriamente
    def colocar_aleatoriamente(self, elemento):
        intentos = 0
        max_intentos = 100  # Establecer un límite de intentos para evitar bucles infinitos
        while intentos < max_intentos:
            i, j = random.randint(0, 5), random.randint(0, 5)
            if self.es_posicion_valida(i, j, elemento):
                self.matrix[i][j] = elemento
                return True
            intentos += 1
        return False  
    
    #Metodos para determinar si se puede mover a ciertas posiciones o no
    def es_posicion_valida(self, i, j, elemento):
        # Verificar si la casilla está vacía
        if self.matrix[i][j] != 0:
            return False
    
        # Verificar que no esté adyacente al agente
        if self.es_adyacente_a_elemento(i, j, 1):
            return False
    
        # Si es un hoyo o el Wumpus, verificar que no esté adyacente al otro
        if elemento in [3, 4]: 
            elemento_a_evitar = 4 if elemento == 3 else 3
            if self.es_adyacente_a_elemento(i, j, elemento_a_evitar):
                return False
    
        return True


    #Metodo para comprobar si esta adyacente a un elemento
    def es_adyacente_a_elemento(self, i, j, elemento):
        direcciones = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        for dx, dy in direcciones:
            ni, nj = i + dx, j + dy
            if 0 <= ni < 6 and 0 <= nj < 6 and self.es_elemento(self.matrix[ni][nj], elemento):
                return True
        return False

    #Metodo adicional para colocar el oro en una posicion fija
    def colocar_oro(self):
        self.matrix[1][2] = 2

    #Metodo para considerar un numero para cada personaje/tipo de casilla del juego
    def es_elemento(self, valor, elemento):
        if elemento == 2:  # Oro
            return valor in [2, 8, 9, 10]
        elif elemento == 3:  # Wumpus
            return valor == 3
        elif elemento == 4:  # Hoyo
            return valor == 4
        elif elemento == 5:  # Hedor
            return valor in [5, 7, 8, 10]
        elif elemento == 6:  # Brisa
            return valor in [6, 7, 9, 10]
        elif elemento == 1:  # Agente 
            return valor == 1 or valor == 11  # Considerar tanto 1 (agente) como 11 (agente + oro)
        else:
            return valor == elemento

    #Metodo para encontrar elemento en tablero (localizar 1 solo elemento)
    def encontrar_elemento(self, elemento):
        for i in range(6):
            for j in range(6):
                if self.es_elemento(self.matrix[i][j], elemento):
                    return [i, j]
        return None

    #Metodo para encontrar elementos en tablero (localizar multiples elementos)
    def encontrar_elementos(self, elementos):
        posiciones = []
        for i in range(6):
            for j in range(6):
                if any(self.es_elemento(self.matrix[i][j], elem) for elem in elementos):
                    posiciones.append([i, j])
        return posiciones
    
    #Metodos para verificar si el agente puede moverse aen una dirección específica
    def canMoveUp(self, row: int, col: int) -> bool:
        if row > 0 and self.matrix[row-1][col] not in [3, 4]:  
            return True
        return False

    def canMoveDown(self, row: int, col: int) -> bool:
        if row < 5 and self.matrix[row+1][col] not in [3, 4]: 
            return True
        return False

    def canMoveLeft(self, row: int, col: int) -> bool:
        if col > 0 and self.matrix[row][col-1] not in [3, 4]:  
            return True
        return False

    def canMoveRight(self, row: int, col: int) -> bool:
        if col < 5 and self.matrix[row][col+1] not in [3, 4]:  
            return True
        return False
    
    #Determina los movimientos disponibles para Max (agente)
    def getAvailableMovesForMax(self, row: int, col: int) -> List[int]:      
        moves = []
        if self.canMoveUp(row, col):
            moves.append(0)
        if self.canMoveDown(row, col):
            moves.append(1)
        if self.canMoveLeft(row, col):
            moves.append(2)
        if self.canMoveRight(row, col):
            moves.append(3)
        return moves
    
    #Determina los movimientos disponibles para Min(hoyos)
    def getAvailableMovesForMin(self) -> List[Tuple[int]]:
        moves = []
        for i in range(6):
            for j in range(6):
                if self.matrix[i][j] == 0:  
                    moves.append((i, j))
        return moves
   
    #Funciones de movimientos en todas las direcciones
    def up(self, row: int, col: int):
        if self.canMoveUp(row, col):
            new_state = Tablerowumpus()
            new_state.setMatrix(self.matrix)
            new_state.matrix[row][col], new_state.matrix[row-1][col] = 0, 1
            #Volver a identicar los avisos de los peligros para que no te salga un 0
            new_state.identificaVecinos()
            return new_state
        return None
  
    def down(self, row: int, col: int):
        if self.canMoveDown(row, col):
            new_state = Tablerowumpus()
            new_state.setMatrix(self.matrix)
            new_state.matrix[row][col], new_state.matrix[row+1][col] = 0, 1
            #Volver a identicar los avisos de los peligros para que no te salga un 0
            new_state.identificaVecinos()
            return new_state
        return None
    
  
    def left(self, row: int, col: int):
        if self.canMoveLeft(row, col):
            new_state = Tablerowumpus()
            new_state.setMatrix(self.matrix)
            new_state.matrix[row][col], new_state.matrix[row][col-1] = 0, 1
            #Volver a identicar los avisos de los peligros para que no te salga un 0
            new_state.identificaVecinos()
            return new_state
        return None

    
    def right(self, row: int, col: int):
        if self.canMoveRight(row, col):
            new_state = Tablerowumpus()
            new_state.setMatrix(self.matrix)
            new_state.matrix[row][col], new_state.matrix[row][col+1] = 0, 1
            #Volver a identicar los avisos de los peligros para que no te salga un 0
            new_state.identificaVecinos()
            return new_state
        return None
    
    #Determina si los jugadores pueden mover
    def moveCanBeMade(self, player: int) -> bool:
        if player == 1:  # Agente
            agent_pos = self.encontrar_elemento(1)
            return len(self.getAvailableMovesForMax(*agent_pos)) > 0
        else:  # Entorno (hoyos)
            return len(self.getAvailableMovesForMin()) > 0
    
    #Metodo con condición de final de juego
    def isGameOver(self) -> bool:
        # Verificar si el agente ha encontrado el oro
        # El agente y el oro estarán en la misma casilla con el valor 11
        pos_agente = self.encontrar_elemento(1)  # Posición del agente
        if self.matrix[pos_agente[0]][pos_agente[1]] == 11:
            return True
        return False 
    
   #Colocar peligro
    def place_danger(self, pos):
        new_state = Tablerowumpus()
        new_state.setMatrix(self.matrix)
        i, j = pos
        if new_state.matrix[i][j] == 0:  # Si la casilla está vacía
            new_state.matrix[i][j] = 4  # Colocar un hoyo
            new_state.marcarVecinos(i, j, 6)  # Marcar las casillas adyacentes con brisa
        return new_state
 
    #---------------------------------------------------------
    #MINIMAX
    #---------------------------------------------------------
    
    #Lo he hecho dentro de la clase tablero
    @staticmethod
    def miniMax(state: 'Tablerowumpus', currentLevel: int, maxLevel: int, player: int, alpha: float, beta: float, stop: bool) -> Tuple['Tablerowumpus', float, bool]:
        
        if state.isGameOver():
            return (state, float('inf'), True)  # Victoria
            
        pos_oro = state.encontrar_elemento(2)

        if currentLevel == maxLevel or not state.moveCanBeMade(player):
            return (state, state.utility(state.encontrar_elemento(1), state.encontrar_elemento(2)), stop)
        
        successorStates = []
    
        if player == 1:  # Agente (MAX)
            agent_pos = state.encontrar_elemento(1)
            moves = state.getAvailableMovesForMax(*agent_pos)
            optimal_moves = []
            all_possible_moves = []
            
            for move in moves:
                if move == 0:
                    new_state = state.up(*agent_pos)
                elif move == 1:
                    new_state = state.down(*agent_pos)
                elif move == 2:
                    new_state = state.left(*agent_pos)
                elif move == 3:
                    new_state = state.right(*agent_pos)
                if new_state:
                    
                    # Comprobar si la nueva posición aumenta la distancia al oro
                    nueva_distancia_oro = abs(new_state.encontrar_elemento(1)[0] - pos_oro[0]) + abs(new_state.encontrar_elemento(1)[1] - pos_oro[1])
                    distancia_actual_oro = abs(agent_pos[0] - pos_oro[0]) + abs(agent_pos[1] - pos_oro[1])

                    # Solo agregar el nuevo estado si no aumenta la distancia al oro
                    if nueva_distancia_oro <= distancia_actual_oro :
                        optimal_moves.append(new_state)
                        
                    all_possible_moves.append(new_state)
                    
            # Si hay movimientos óptimos,los uso; de lo contrario, usa todos los movimientos posibles
            successorStates = optimal_moves if optimal_moves else all_possible_moves
                                             
        else:  # Entorno (MIN)
            moves = state.getAvailableMovesForMin()
            for move in moves:
                new_state = Tablerowumpus()
                new_state.setMatrix(state.getMatrix())
                if new_state.matrix[move[0]][move[1]] == 0:
                    new_state.matrix[move[0]][move[1]] = 4  # Colocar un hoyo
                    new_state.marcarVecinos(move[0], move[1], 6)  # Marcar las casillas adyacentes con brisa
                successorStates.append(new_state)

        if len(successorStates) == 0:
            return (state, state.utility(state.encontrar_elemento(1), state.encontrar_elemento(2)), True)

        bestState = None
    
        if player == 1:  # Agente (MAX)
            maxValue = float('-inf')
            for successor in successorStates:
                _, utility, new_stop = Tablerowumpus.miniMax(successor, currentLevel + 1, maxLevel, 2, alpha, beta, stop)
                if utility > maxValue:
                    maxValue = utility
                    bestState = successor
                alpha = max(alpha, utility)
                if utility >= beta:
                    return (successor, utility, new_stop)
            return (bestState, maxValue, stop)
        else:  # Entorno (MIN)
            minValue = float('inf')
            for successor in successorStates:
                _, utility, new_stop = Tablerowumpus.miniMax(successor, currentLevel + 1, maxLevel, 1, alpha, beta, stop)
                if utility < minValue:
                    minValue = utility
                    bestState = successor
                beta = min(beta, utility)
                if utility <= alpha:
                    return (successor, utility, new_stop)
            return (bestState, minValue, stop)

    #Funcion que llama al algoritmo minimax 
    @staticmethod
    def performActionMinMax(state: 'Tablerowumpus', player: int):
        
        if state.isGameOver():
            return "victoria", state.getMatrix()
        
        matrizB = state.getMatrix()
        tmpMatriz = [row[:] for row in matrizB]

        # Profundidad recomendada
        depth = 2

        matrizoptima = tmpMatriz
        stop = False 
        currentLevel = 0    

        if player == 1:  # Agente
            pos_agente = state.encontrar_elemento(1)
            pos_oro = state.encontrar_elemento(2)
        
            # Verificar si el oro está adyacente
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                if [pos_agente[0] + dx, pos_agente[1] + dy] == pos_oro:
                    # Mover directamente al oro si está adyacente
                    new_state = Tablerowumpus()
                    new_state.setMatrix(state.getMatrix())
    
                    # Elimina el agente de su posición anterior, pero no el oro
                    new_state.matrix[pos_agente[0]][pos_agente[1]] = 0
    
                    # Coloca tanto al agente como el oro en la misma casilla (usando el valor 11)
                    new_state.matrix[pos_oro[0]][pos_oro[1]] = 11  # Valor 11 representa el agente con el oro
    
                    return "mover_a_oro", new_state.getMatrix()
                
        
            # Si el oro no está adyacente, continuar con el algoritmo Minimax normal
            tmpMatrizB = Tablerowumpus()
            tmpMatrizB.setMatrix(tmpMatriz)
            (tablerowumpus_optimo, valoroptimo, stop) = Tablerowumpus.miniMax(tmpMatrizB, currentLevel, depth, player, -float('inf'), float('inf'), stop)

            if tablerowumpus_optimo is None:
                return "ninguno", state.getMatrix()
            
            matrizoptima = tablerowumpus_optimo.getMatrix()  # Obtener la matriz del objeto Tablerowumpus

            # Encontrar el movimiento realizado
            agente_viejo = state.encontrar_elemento(1)
            agente_nuevo = next(([i, j] for i in range(6) for j in range(6) if matrizoptima[i][j] == 1), None)

            # Determinar la dirección del movimiento
            if agente_nuevo:
                if agente_nuevo[0] < agente_viejo[0]:
                    movimiento = "arriba"
                elif agente_nuevo[0] > agente_viejo[0]:
                    movimiento = "abajo"
                elif agente_nuevo[1] < agente_viejo[1]:
                    movimiento = "izquierda"
                elif agente_nuevo[1] > agente_viejo[1]:
                    movimiento = "derecha"
                else:
                    movimiento = "ninguno"
            else:
                movimiento = "ninguno"

            return movimiento, matrizoptima

        else:  #  Entorno (mueve hoyos aleatoriamente)
            hoyos = state.encontrar_elementos([4])  # Encontrar todos los hoyos
            if hoyos:
                hoyo_a_mover = random.choice(hoyos)
                casillas_vacias = [(i, j) for i in range(6) for j in range(6) if matrizB[i][j] == 0]
                
                if casillas_vacias:
                    nueva_pos = random.choice(casillas_vacias)
                
                    # Eliminar el hoyo de su posición actual
                    matrizoptima[hoyo_a_mover[0]][hoyo_a_mover[1]] = 0
                
                    # Actualizar brisas de la posición anterior
                    Tablerowumpus.actualizarBrisas(matrizoptima, hoyo_a_mover[0], hoyo_a_mover[1], hoyos)
                
                    # Colocar el hoyo en la nueva posición
                    matrizoptima[nueva_pos[0]][nueva_pos[1]] = 4
                
                    # Añadir nuevas brisas
                    nuevo_tablero = Tablerowumpus(matrix=matrizoptima)
                    nuevo_tablero.marcarVecinos(nueva_pos[0], nueva_pos[1], 6)
                    matrizoptima = nuevo_tablero.getMatrix()
    
            return "mover_hoyo", matrizoptima   
    
    #Metodo encargado de asegurarse de que se actualicen los avisos de los peligros de forma optima tras mover hoyos
    #Mi idea era hacer una funcion especifica para cuando se moviesen los hoyos para poder 
    #manejar los avisos de donde desaparecen y de donde aparece el nuevo hoyo
    #Aun asi, soy consciente de que tiene parte de repetición de codigo con las funciones de marcarvecinos
    #Si lo hiciese de nuevo no lo haria asi, pero ahora mismo no estoy en condición de cambiarlo
    @staticmethod
    def actualizarBrisas(matriz, fila, columna, hoyos):
        direcciones = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
        def actualizar_casilla(f, c):
            if 0 <= f < 6 and 0 <= c < 6:
                elemento_actual = matriz[f][c]
                necesita_brisa = any(matriz[f + dx][c + dy] == 4 
                                     for dx, dy in direcciones 
                                     if 0 <= f + dx < 6 and 0 <= c + dy < 6)
                necesita_hedor = matriz[f][c] == 3  # Verifica si hay Wumpus al lado
                tiene_oro = matriz[f][c] in [2, 8, 9, 10]  # Verifica si la casilla contiene oro
                
                # No modificar la posición del agente
                if elemento_actual == 1:
                    return
               
                #Para manejar el caso de que haya 2 hoyos adyacentes
                if elemento_actual == 4:
                    if necesita_brisa and necesita_hedor:
                        if tiene_oro:
                            matriz[f][c] = 10  # Oro + Brisa + Hedor
                        else:
                            matriz[f][c] = 7   # Brisa + Hedor
                    elif necesita_brisa:
                        if tiene_oro:
                            matriz[f][c] = 9   # Oro + Brisa
                        else:
                            matriz[f][c] = 6   # Solo Brisa
                    return
                                   
                if not necesita_brisa and not necesita_hedor:
                    if tiene_oro:
                        matriz[f][c] = 2  
                    elif matriz[f][c] == 6:
                        matriz[f][c] = 0
                    elif matriz[f][c] == 7:
                        matriz[f][c] = 5
                    elif matriz[f][c] == 9:
                        matriz[f][c] = 2
                    elif matriz[f][c] == 10:
                        matriz[f][c] = 8
                elif necesita_brisa and not necesita_hedor:
                    if tiene_oro:
                        matriz[f][c] = 9  # Oro + Brisa
                    else:
                        matriz[f][c] = 6
                elif not necesita_brisa and necesita_hedor:
                    if tiene_oro:
                        matriz[f][c] = 8  # Oro + Hedor
                    else:
                        matriz[f][c] = 5
                elif necesita_brisa and necesita_hedor:
                    if tiene_oro:
                        matriz[f][c] = 10  # Oro + Brisa + Hedor
                    else:
                        matriz[f][c] = 7

        # Actualizar la posición que deja el hoyo y sus adyacentes
        actualizar_casilla(fila, columna)
        for dx, dy in direcciones:
            actualizar_casilla(fila + dx, columna + dy)    

        
    @staticmethod
    def AIAction(state: 'Tablerowumpus', player: int):
        global AIReadyToMove

        matriz = state.getMatrix()
    
        # Llamada a performActionMinMax
        movimiento, matriz_optima = Tablerowumpus.performActionMinMax(Tablerowumpus(matriz), player)
    
        AIReadyToMove = False
    
        # Actualizar el estado del juego con el movimiento óptimo
        if player == 1:  # Agente
            if movimiento != "ninguno":
                pos_actual = state.encontrar_elemento(1)
                if movimiento == "arriba":
                    nueva_pos = [pos_actual[0] - 1, pos_actual[1]]
                elif movimiento == "abajo":
                    nueva_pos = [pos_actual[0] + 1, pos_actual[1]]
                elif movimiento == "izquierda":
                    nueva_pos = [pos_actual[0], pos_actual[1] - 1]
                elif movimiento == "derecha":
                    nueva_pos = [pos_actual[0], pos_actual[1] + 1]
                
                state.matrix[pos_actual[0]][pos_actual[1]] = 0
                state.matrix[nueva_pos[0]][nueva_pos[1]] = 1
                   
                 # Restaurar peligros después del movimiento
                state.identificaVecinos()  # Actualiza brisas y hedores en el tablero
                
                # Mover un hoyo aleatoriamente
                hoyos = state.encontrar_elementos([4])  # Encontrar todos los hoyos
                if hoyos:
                    hoyo_a_mover = random.choice(hoyos)
                    estado_hoyo_x, estado_hoyo_y = hoyo_a_mover
                
                    # Encontrar casillas vacías para mover el hoyo
                    casillas_vacias = [(i, j) for i in range(6) for j in range(6) if state.matrix[i][j] == 0 and not state.es_adyacente_a_elemento(i, j, 3)]  # Asegúrate de que no esté cerca del Wumpus
                
                    if casillas_vacias:
                        nueva_pos_hoyo = random.choice(casillas_vacias)
                    
                        # Quitar brisas de la posición anterior del hoyo
                        state.marcarVecinos(estado_hoyo_x, estado_hoyo_y, 0)  # Quitar brisas antiguas
                    
                        # Eliminar el hoyo de su posición actual
                        state.matrix[estado_hoyo_x][estado_hoyo_y] = 0  
                    
                        # Colocar el hoyo en la nueva posición y añadir brisas
                        state.matrix[nueva_pos_hoyo[0]][nueva_pos_hoyo[1]] = 4  # Colocar el hoyo en la nueva posición
                        state.marcarVecinos(nueva_pos_hoyo[0], nueva_pos_hoyo[1], 6)  # Añadir nuevas brisas
    
        else:  # Entorno (mover hoyo)
            state.setMatrix(matriz_optima)

        return movimiento, state         
                       

In [ ]:

#Metodo que aplica la jugabilidad
def jugar_wumpus():
    tablero = Tablerowumpus()
    ronda = 0  # Inicializamos en 0 para el tablero inicial
    juego_terminado = False

    # Mostrar el tablero inicial
    print("Tablero inicial:")
    tablero.printTablero()
    pos_agente = tablero.encontrar_elemento(1)
    pos_oro = tablero.encontrar_elemento(2)
    utilidad = tablero.utility(pos_agente, pos_oro)
    
    # Descripción de los elementos del juego
    print("Descripción de los elementos del tablero\n")
    print("1 --> Agente")
    print("2 --> Oro")
    print("3 --> Wumpus")
    print("4 --> Hoyo")
    print("5 --> Hedor")
    print("6 --> Brisa")
    print("7 --> Hedor + Brisa")
    print("8 --> Oro + Hedor")
    print("9 --> Oro + Brisa")
    print("10 --> Oro + Hedor + Brisa")
    print("11 --> AGENTE LLEGA AL ORO")
    print("\n¡Comienza el juego!\n")

    while not juego_terminado:
        ronda += 1
        
        accion = input("Presiona Enter para el turno del Agente o 's' para salir: ")
        if accion.lower() == 's':
            print("Juego terminado por el usuario (antes de finalizar).")
            return

        print(f"RONDA {ronda}")
        
        # Turno del Agente (Minimax)
        print(f"TURNO AGENTE")
        movimiento, matriz_optima = Tablerowumpus.performActionMinMax(tablero, 1)
        tablero.setMatrix(matriz_optima)
        
        tablero.printTablero()
        pos_agente = tablero.encontrar_elemento(1)
        pos_oro = tablero.encontrar_elemento(2)
        utilidad = tablero.utility(pos_agente, pos_oro)

        # Verificar si el juego ha terminado
        if tablero.isGameOver():
            print("¡El agente ha encontrado el oro! ¡Victoria!")
            juego_terminado = True
            break

        accion = input("Presiona Enter para el turno del Entorno o 's' para salir: ")
        if accion.lower() == 's':
            print("Juego terminado por el usuario (antes de finalizar).")
            return
        
        # Turno del Entorno (Movimiento aleatorio de hoyo)
        print(f"TURNO ENTORNO")
        movimiento, matriz_optima = Tablerowumpus.performActionMinMax(tablero, 2)
        tablero.setMatrix(matriz_optima)

        tablero.printTablero()
        pos_agente = tablero.encontrar_elemento(1)
        pos_oro = tablero.encontrar_elemento(2)
        utilidad = tablero.utility(pos_agente, pos_oro)

if __name__ == "__main__":
    jugar_wumpus()
    
  




Tablero inicial:
0 6 4 6 0 0
0 0 9 4 6 5
0 0 0 6 5 3
0 0 0 0 0 5
0 0 0 0 0 0
1 0 0 0 0 0

Descripción de los elementos del tablero

1 --> Agente
2 --> Oro
3 --> Wumpus
4 --> Hoyo
5 --> Hedor
6 --> Brisa
7 --> Hedor + Brisa
8 --> Oro + Hedor
9 --> Oro + Brisa
10 --> Oro + Hedor + Brisa
11 --> AGENTE LLEGA AL ORO

¡Comienza el juego!

